# Day 3 AM

- lexical dispersion
- more semantic networks with custom dictionaries
- LIWC in German

In [ ]:
import re
import os
import pandas
import numpy
import matplotlib
import matplotlib.pyplot as plt
import nltk
import sotu
import networkx
import wordcloud
import scipy.cluster.hierarchy
import scipy.spatial.distance
import sklearn.feature_extraction.text
import sklearn.decomposition
from itertools import combinations

matplotlib.style.use("ggplot")
matplotlib.rcParams["axes.facecolor"] = "#EBEBEB"
matplotlib.rcParams["axes.edgecolor"] = "#EBEBEB"
matplotlib.rcParams["axes.labelcolor"] = "#3C3C3C"
matplotlib.rcParams["xtick.color"] = "#3C3C3C"
matplotlib.rcParams["ytick.color"] = "#3C3C3C"
matplotlib.rcParams["grid.color"] = "white"
matplotlib.rcParams["axes.grid"] = True

nltk.download("stopwords", quiet=True)

sotu_df = sotu.load().reset_index(drop=True)

In [ ]:
def load_liwc(path):
    text = open(path, encoding="utf-8", errors="replace").read()
    blocks = text.split("%", 2)
    categories = {}
    for line in blocks[1].strip().splitlines():
        parts = line.split("\t")
        if len(parts) >= 2 and parts[0].strip().isdigit():
            categories[int(parts[0])] = parts[1].strip()
    exact = {}
    trie = {}
    for line in blocks[2].strip().splitlines():
        parts = line.split("\t")
        if len(parts) < 2:
            continue
        word = parts[0].strip().lower()
        if " " in word or word.startswith("("):
            continue
        cats = []
        for c in parts[1:]:
            c = c.strip()
            if c.isdigit() and int(c) in categories:
                cats.append(int(c))
        if not cats:
            continue
        if word.endswith("*"):
            node = trie
            for ch in word[:-1]:
                node = node.setdefault(ch, {})
            node["$"] = cats
        else:
            exact[word] = cats
    return categories, exact, trie

def liwc_match(token, exact, trie):
    if token in exact:
        return exact[token]
    node = trie
    best = None
    for ch in token:
        if "$" in node:
            best = node["$"]
        if ch not in node:
            return best or []
        node = node[ch]
    if "$" in node:
        best = node["$"]
    return best or []

def score_with_dict(text, categories, exact, trie, token_pattern=r"[A-Za-z']+"):
    tokens = re.findall(token_pattern, text.lower())
    wc = len(tokens)
    if wc == 0:
        return None
    counts = {cat: 0 for cat in categories.values()}
    for tok in tokens:
        for c in liwc_match(tok, exact, trie):
            counts[categories[c]] += 1
    return {cat: 100 * counts[cat] / wc for cat in categories.values()}

In [ ]:
nuke_cats, nuke_exact, nuke_trie = load_liwc("../day-1/dictionaries/nuke.dic")
print(f"nuke.dic: {len(nuke_cats)} categories, {len(nuke_exact)} exact words")

rows = []
for i in range(len(sotu_df)):
    speech = sotu_df.iloc[i]
    scored = score_with_dict(speech["text"], nuke_cats, nuke_exact, nuke_trie)
    if scored is None:
        continue
    scored["year"] = speech["year"]
    scored["president"] = speech["president"]
    rows.append(scored)
nuke_df = pandas.DataFrame(rows)

In [ ]:
def plot_timeseries(category, df, ylabel=None):
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(df["year"], df[category],
            marker="o", color="black", linewidth=0.8, markersize=4)
    ax.set_xlabel("year")
    ax.set_ylabel(ylabel or category)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
def lexical_dispersion_plot(documents, patterns, title=None,
                            token_pattern=r"[A-Za-z]+"):
    """Vertical-tick lexical dispersion plot.
    documents: list of (label, text) tuples.
    patterns: list of search terms; trailing '*' means prefix match.
    """
    n_terms = len(patterns)
    n_docs = len(documents)
    fig, axes = plt.subplots(1, n_terms,
                              figsize=(4 * n_terms, max(3, 0.3 * n_docs + 1)),
                              sharey=True)
    if n_terms == 1:
        axes = [axes]
    for ax, pattern in zip(axes, patterns):
        prefix = pattern.rstrip("*").lower()
        is_wild = pattern.endswith("*")
        ax.set_title(pattern, backgroundcolor="#cfcfcf", fontsize=10)
        for i, (label, text) in enumerate(documents):
            tokens = re.findall(token_pattern, text.lower())
            ax.barh(i, len(tokens), color="#e0e0e0", height=0.85, zorder=0)
            positions = []
            for j, tok in enumerate(tokens):
                hit = tok.startswith(prefix) if is_wild else tok == prefix
                if hit:
                    positions.append(j)
            if positions:
                ax.vlines(positions, i - 0.4, i + 0.4,
                          color="black", linewidth=0.6, zorder=2)
        ax.set_yticks(range(n_docs))
        ax.set_yticklabels([d[0] for d in documents], fontsize=7)
        ax.set_xlabel("Token index")
        ax.invert_yaxis()
        ax.grid(axis="x", color="white", zorder=1)
    if title:
        fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
def category_network(df, categories, ax=None, title=None,
                     min_weight=0.0, scale=4000):
    """Co-occurrence network from a per-document category-frequency DataFrame.
    Nodes sized by mean prevalence; edges weighted by Pearson correlation.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8))
    G = networkx.Graph()
    for c in categories:
        prev = df[c].mean() if c in df.columns else 0.0
        G.add_node(c, prevalence=prev)
    present = [c for c in categories if c in df.columns]
    if len(present) >= 2 and len(df) >= 2:
        corr = df[present].corr().fillna(0.0)
        for a, b in combinations(present, 2):
            w = max(corr.loc[a, b], min_weight)
            G.add_edge(a, b, weight=w)
    pos = networkx.circular_layout(G)
    sizes = [G.nodes[n]["prevalence"] * scale + 200 for n in G.nodes]
    widths = [G[u][v]["weight"] * 4 for u, v in G.edges]
    networkx.draw_networkx_nodes(G, pos, node_color="#FFC0CB",
                                 node_size=sizes, ax=ax,
                                 edgecolors="#A06070", linewidths=0.5)
    networkx.draw_networkx_labels(G, pos, font_size=8, ax=ax)
    networkx.draw_networkx_edges(G, pos, width=widths,
                                 edge_color="#888888", alpha=0.6, ax=ax)
    ax.set_title(title or "")
    ax.axis("off")

nuke_category_list = list(nuke_cats.values())

## State of the Union addresses

- From 1790 to present
- Yearly (unlike inaugurals)
- Mixed format (written vs spoken)
- Varying lengths (1k-35k)
- Available natively in the `sotu` Python package
- Plus some fun metadata

The promise of 'nuclear' (and 'atomic' and 'nuke') in SOTUs

<div class="image-grid">

![Cooling towers of a nuclear power plant.](img/p101-i1.png)
![Mushroom cloud from a nuclear weapon test.](img/p101-i5.png)
![Albert Einstein at a blackboard with E=mc^2.](img/p101-i0.png)
![J. Robert Oppenheimer with the quote: "Now I am become death, destroyer of worlds."](img/p101-i6.png)

</div>

## Before we get into the analysis…

Let's talk about **bigrams** and other **n-grams**.

## Bigrams in the SOTU corpus

In [ ]:
vectorizer = sklearn.feature_extraction.text.CountVectorizer(
    ngram_range=(2, 2), stop_words="english", min_df=20)
X = vectorizer.fit_transform(sotu_df["text"])
totals = numpy.asarray(X.sum(axis=0)).flatten()
feats = vectorizer.get_feature_names_out()
bigram_counts = pandas.Series(totals, index=feats).sort_values(ascending=False)
print(bigram_counts.head(20).to_string())

## But what do American presidents talk about when they talk about 'nuclear' (and not about non-nuclear)?

In [ ]:
# Build two sub-corpora: SOTUs containing 'nuclear*' vs the rest.
nuclear_mask = sotu_df["text"].str.lower().str.contains(r"nuclear\w*", regex=True)
vec = sklearn.feature_extraction.text.CountVectorizer(
    stop_words="english", min_df=5, token_pattern=r"[A-Za-z][A-Za-z\-]+")
Xall = vec.fit_transform(sotu_df["text"])
feats = vec.get_feature_names_out()
in_target = Xall[nuclear_mask.values].sum(axis=0).A1
in_ref = Xall[~nuclear_mask.values].sum(axis=0).A1
n_t = in_target.sum()
n_r = in_ref.sum()
# Chi^2 keyness
expected_t = (in_target + in_ref) * n_t / (n_t + n_r)
expected_r = (in_target + in_ref) * n_r / (n_t + n_r)
chi2 = ((in_target - expected_t) ** 2 / (expected_t + 1)
        + (in_ref - expected_r) ** 2 / (expected_r + 1))
sign = numpy.sign(in_target / max(n_t, 1) - in_ref / max(n_r, 1))
keyness = pandas.DataFrame({
    "feature": feats,
    "chi2": chi2 * sign,
    "n_target": in_target,
    "n_reference": in_ref,
}).sort_values("chi2", ascending=False)
print(keyness.head(30).to_string(index=False))

## fodder for the `nuke.dic` dictionary

## Lexical dispersion: 'atom*' across modern SoUs

In [ ]:
modern = sotu_df[sotu_df["year"] >= 1940].reset_index(drop=True)
documents = [(f"{r['president']}-{r['year']}", r["text"])
             for _, r in modern.iterrows()]
# Subsample to keep the plot readable
lexical_dispersion_plot(documents[::3], ["atom*"], title="Lexical dispersion plot")

## Lexical dispersion: 'nuclear*' across modern SoUs

In [ ]:
lexical_dispersion_plot(documents[::3], ["nuclear*"], title="Lexical dispersion plot")

## LIWCing at 'nuke's

In [ ]:
plot_timeseries("nuke", nuke_df)

## Semantic network (whole SOTU corpus)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
category_network(nuke_df, nuke_category_list, ax=ax,
                 title="Whole SOTU corpus")
plt.tight_layout()
plt.show()

## Semantic network (1940s SOTU corpus)

<div class="side-by-side">

![Harry Truman, official portrait against an American flag backdrop.](img/p110-i1.png)

</div>

> I'm a war criminal! :)

In [ ]:
sub = nuke_df[(nuke_df["year"] >= 1940) & (nuke_df["year"] < 1950)]
fig, ax = plt.subplots(figsize=(10, 6))
category_network(sub, nuke_category_list, ax=ax,
                 title="1940s SOTUs (n={})".format(len(sub)))
plt.tight_layout()
plt.show()

## Semantic network (1950s SOTU corpus)

<div class="side-by-side">

![Dwight D. Eisenhower at a microphone.](img/p111-i1.png)

</div>

> I tried to warn you about the military-industrial complex. :(

In [ ]:
sub = nuke_df[(nuke_df["year"] >= 1950) & (nuke_df["year"] < 1960)]
fig, ax = plt.subplots(figsize=(10, 6))
category_network(sub, nuke_category_list, ax=ax,
                 title="1950s SOTUs (n={})".format(len(sub)))
plt.tight_layout()
plt.show()

## Semantic network (1960s SOTU corpus)

<div class="side-by-side">

![Lyndon B. Johnson at his desk, hand on forehead.](img/p112-i1.png)

</div>

> I'm a war criminal. :(

In [ ]:
sub = nuke_df[(nuke_df["year"] >= 1960) & (nuke_df["year"] < 1970)]
fig, ax = plt.subplots(figsize=(10, 6))
category_network(sub, nuke_category_list, ax=ax,
                 title="1960s SOTUs (n={})".format(len(sub)))
plt.tight_layout()
plt.show()

## Semantic network (1970s SOTU corpus)

<div class="side-by-side">

![Richard Nixon (left) and Henry Kissinger (right) in the Oval Office.](img/p113-i1.png)

</div>

> We're war criminals! :)

In [ ]:
sub = nuke_df[(nuke_df["year"] >= 1970) & (nuke_df["year"] < 1980)]
fig, ax = plt.subplots(figsize=(10, 6))
category_network(sub, nuke_category_list, ax=ax,
                 title="1970s SOTUs (n={})".format(len(sub)))
plt.tight_layout()
plt.show()

## Semantic network (1980s SOTU corpus)

<div class="side-by-side">

![Ronald Reagan (left) and George H. W. Bush (right) at the White House.](img/p114-i1.png)

</div>

> We're war criminals! :)

In [ ]:
sub = nuke_df[(nuke_df["year"] >= 1980) & (nuke_df["year"] < 1990)]
fig, ax = plt.subplots(figsize=(10, 6))
category_network(sub, nuke_category_list, ax=ax,
                 title="1980s SOTUs (n={})".format(len(sub)))
plt.tight_layout()
plt.show()

## Semantic network (1990s SOTU corpus)

<div class="side-by-side">

![Bill Clinton, official portrait against an American flag.](img/p115-i1.png)

</div>

> I'm a war criminal and a sex pest! :)

In [ ]:
sub = nuke_df[(nuke_df["year"] >= 1990) & (nuke_df["year"] < 2000)]
fig, ax = plt.subplots(figsize=(10, 6))
category_network(sub, nuke_category_list, ax=ax,
                 title="1990s SOTUs (n={})".format(len(sub)))
plt.tight_layout()
plt.show()

## Semantic network (2000s SOTU corpus)

<div class="side-by-side">

![Colin Powell (left), George W. Bush (centre), and Dick Cheney (right).](img/p116-i1.png)

</div>

> We're war criminals! :)

In [ ]:
sub = nuke_df[(nuke_df["year"] >= 2000) & (nuke_df["year"] < 2010)]
fig, ax = plt.subplots(figsize=(10, 6))
category_network(sub, nuke_category_list, ax=ax,
                 title="2000s SOTUs (n={})".format(len(sub)))
plt.tight_layout()
plt.show()

## Semantic network (2010s+ SOTU corpus)

<div class="side-by-side">

![Donald Trump (left) speaking with Barack Obama (right).](img/p117-i1.png)

</div>

> We're war criminals! :)

In [ ]:
sub = nuke_df[(nuke_df["year"] >= 2010) & (nuke_df["year"] < 2030)]
fig, ax = plt.subplots(figsize=(10, 6))
category_network(sub, nuke_category_list, ax=ax,
                 title="2010s+ SOTUs (n={})".format(len(sub)))
plt.tight_layout()
plt.show()

In conclusion, <span class="color-red">concepts</span> — or at least their inferential and associative connections in relevant **corpora** — do **change** in measurable ways.

![A head on a pike, in case the 'war criminal' line wasn't on the nose enough.](img/p118-i1.png)

But one thing that doesn't change is that American presidents are war criminals. :)

# And now for something completely different…

![John Cleese seated at a desk in the surf, captioned "And now for something completely different…".](img/p119-i0.png)

# Friedrich Nietzsche

Digital Critical Edition (eKGWB) — Colli & Montinari, ed. D'Iorio

![Nietzschesource.org — Friedrich Nietzsche Digitale Kritische Gesamtausgabe.](img/p120-i0.png)

In [ ]:
nietzsche_docs = []
for fname in sorted(os.listdir("nietzsche")):
    if not fname.endswith(".txt"):
        continue
    with open(f"nietzsche/{fname}", encoding="utf-8", errors="replace") as f:
        text = f.read()
    parts = fname.replace(".txt", "").split(" ", 1)
    year = int(parts[0])
    abbrev = parts[1]
    nietzsche_docs.append({"year": year, "abbrev": abbrev,
                           "label": f"{year} {abbrev}",
                           "text": text,
                           "tokens": len(re.findall(r"[a-zäöüß']+", text.lower()))})
nietzsche_df = pandas.DataFrame(nietzsche_docs)
print(f"loaded {len(nietzsche_df)} Nietzsche works, {nietzsche_df['tokens'].sum()} tokens total")

## Nietzsche works — Year × Tokens

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(nietzsche_df["year"], nietzsche_df["tokens"],
           s=0, color="black")
for _, r in nietzsche_df.iterrows():
    ax.annotate(r["abbrev"], (r["year"], r["tokens"]),
                ha="center", va="center",
                bbox=dict(boxstyle="round,pad=0.3",
                          facecolor="white", edgecolor="black"))
ax.set_xlabel("Year")
ax.set_ylabel("Tokens")
plt.tight_layout()
plt.show()

In [ ]:
german_stopwords = set(nltk.corpus.stopwords.words("german"))

def tokenize_german(text):
    tokens = re.findall(r"[a-zäöüß']+", text.lower())
    return [t for t in tokens if t not in german_stopwords and len(t) > 2]

vec = sklearn.feature_extraction.text.CountVectorizer(
    tokenizer=tokenize_german, token_pattern=None, min_df=2)
dfm_nietzsche = vec.fit_transform(nietzsche_df["text"])
ny_features = vec.get_feature_names_out()
totals_ny = numpy.asarray(dfm_nietzsche.sum(axis=0)).flatten()
top_ny = pandas.Series(totals_ny, index=ny_features).sort_values(ascending=False)

## Nietzsche corpus — word cloud

In [ ]:
wc = wordcloud.WordCloud(width=1200, height=700, background_color="white",
                          colormap="tab10", random_state=42, max_words=300)
wc.generate_from_frequencies(top_ny.head(300).to_dict())
fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(wc, interpolation="bilinear"); ax.axis("off")
plt.tight_layout(); plt.show()

## Top features in the Nietzsche corpus

In [ ]:
print(top_ny.head(20).to_string())

## Topic modeling using Latent Dirichlet Allocation

In [ ]:
lda = sklearn.decomposition.LatentDirichletAllocation(
    n_components=7, random_state=42, max_iter=20)
lda.fit(dfm_nietzsche)
topics = {}
for k, weights in enumerate(lda.components_):
    idx = weights.argsort()[-10:][::-1]
    topics[f"Topic {k+1}"] = [ny_features[i] for i in idx]
print(pandas.DataFrame(topics).to_string())

## Per-work top features

In [ ]:
for i, row in nietzsche_df.iterrows():
    counts = numpy.asarray(dfm_nietzsche[i].todense()).flatten()
    s = pandas.Series(counts, index=ny_features).sort_values(ascending=False)
    print(f"\n{row['label']}:")
    print(s.head(8).to_string())

## Hierarchical clustering — Euclidean distance on normalised token frequency

In [ ]:
dfm_dense = dfm_nietzsche.toarray().astype(float)
row_totals = dfm_dense.sum(axis=1, keepdims=True)
row_totals[row_totals == 0] = 1.0
norm = dfm_dense / row_totals
distances = scipy.spatial.distance.pdist(norm, metric="euclidean")
linkage = scipy.cluster.hierarchy.linkage(distances, method="average")
fig, ax = plt.subplots(figsize=(12, 6))
scipy.cluster.hierarchy.dendrogram(linkage,
    labels=nietzsche_df["label"].values, leaf_rotation=90, ax=ax)
ax.set_title("Euclidean Distance on Normalized Token Frequency")
plt.tight_layout(); plt.show()

## Comparison word cloud — early cluster

In [ ]:
clusters = scipy.cluster.hierarchy.fcluster(linkage, t=2, criterion="maxclust")
early_idx = numpy.where(clusters == clusters[0])[0]
late_idx = numpy.where(clusters != clusters[0])[0]

def comparison_cloud(indices):
    n = len(indices)
    cols = min(4, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows))
    axes = numpy.atleast_2d(axes).flatten()
    for k, i in enumerate(indices):
        counts = numpy.asarray(dfm_nietzsche[i].todense()).flatten()
        freqs = dict(zip(ny_features, counts))
        freqs = {w: c for w, c in freqs.items() if c > 0}
        wc = wordcloud.WordCloud(width=400, height=300, background_color="white",
                                  colormap="tab10", random_state=42, max_words=60)
        wc.generate_from_frequencies(freqs)
        axes[k].imshow(wc, interpolation="bilinear")
        axes[k].axis("off")
        axes[k].set_title(nietzsche_df.iloc[i]["label"], fontsize=9)
    for k in range(len(indices), len(axes)):
        axes[k].axis("off")
    plt.tight_layout(); plt.show()

comparison_cloud(early_idx)

## Comparison word cloud — late cluster

In [ ]:
comparison_cloud(late_idx)

# German lexical dispersion

## shame — scham* / schmach* / schand*

In [ ]:
documents = list(zip(nietzsche_df["label"], nietzsche_df["text"]))
lexical_dispersion_plot(documents,
                        ["scham*", "schmach*", "schand*"],
                        token_pattern=r"[a-zäöüß]+")

## trust & mistrust — vertrau* / misstrau*

In [ ]:
lexical_dispersion_plot(documents,
                        ["vertrau*", "misstrau*"],
                        token_pattern=r"[a-zäöüß]+")

In [ ]:
ny_cats, ny_exact, ny_trie = load_liwc("../day-1/dictionaries/nietzsche.dic")
print(f"nietzsche.dic: {len(ny_cats)} categories")

ny_rows = []
for _, r in nietzsche_df.iterrows():
    scored = score_with_dict(r["text"], ny_cats, ny_exact, ny_trie,
                              token_pattern=r"[a-zäöüß]+")
    if scored is None:
        continue
    text_lc = r["text"].lower()
    toks = re.findall(r"[a-zäöüß]+", text_lc)
    sixltr = 100 * sum(1 for t in toks if len(t) >= 6) / max(len(toks), 1)
    scored["Sixltr"] = sixltr
    scored["year"] = r["year"]
    scored["abbrev"] = r["abbrev"]
    scored["label"] = r["label"]
    ny_rows.append(scored)
ny_scored = pandas.DataFrame(ny_rows)
print("Categories scored:", ", ".join(c for c in ny_cats.values())[:200])

In [ ]:
def scatter_works(category, df=None):
    df = df if df is not None else ny_scored
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(df["year"], df[category], s=0)
    for _, r in df.iterrows():
        ax.annotate(r["abbrev"], (r["year"], r[category]),
                    ha="center", va="center",
                    bbox=dict(boxstyle="round,pad=0.3",
                              facecolor="white", edgecolor="black"))
    ax.set_xlabel("Year")
    ax.set_ylabel(category)
    plt.xticks(rotation=45)
    plt.tight_layout(); plt.show()

## Six-letter-word percentage by year

In [ ]:
scatter_works("Sixltr")

## instinct% by year

In [ ]:
# Use whichever instinct-like category exists in nietzsche.dic
cat = next((c for c in ny_cats.values() if c.lower().startswith("instinct")), None)
if cat:
    scatter_works(cat)
else:
    print("no instinct category found")

## shame% by year

In [ ]:
cat = next((c for c in ny_cats.values() if c.lower().startswith("shame")), None)
if cat:
    scatter_works(cat)
else:
    print("no shame category found")

## virtue% by year

In [ ]:
cat = next((c for c in ny_cats.values() if c.lower().startswith("virtue")), None)
if cat:
    scatter_works(cat)
else:
    print("no virtue category found")

# nietzsche.dic semantic networks

Mark Alfano, *Nietzsche's Moral Psychology* (Cambridge UP)

<div class="side-by-side">

![Cover of Mark Alfano's *Nietzsche's Moral Psychology*.](img/p136-i0.png)
![Force-directed network of nietzsche.dic categories with 'life' at the centre.](img/p136-i1.png)

</div>

## Circle (book): nietzsche.dic semantic network

In [ ]:
ny_category_list = [c for c in ny_cats.values() if c in ny_scored.columns]
fig, ax = plt.subplots(figsize=(12, 8))
category_network(ny_scored, ny_category_list, ax=ax,
                 title="Whole Nietzsche corpus", scale=8000)
plt.tight_layout(); plt.show()

In [ ]:
def egonet(focus, df=None, categories=None, ax=None, title=None):
    df = df if df is not None else ny_scored
    categories = categories if categories is not None else ny_category_list
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 7))
    if focus not in df.columns:
        ax.text(0.5, 0.5, f"category '{focus}' not in nietzsche.dic",
                ha="center", va="center")
        ax.axis("off")
        return
    G = networkx.Graph()
    for c in categories:
        G.add_node(c, prevalence=df[c].mean() if c in df.columns else 0.0)
    corr = df[categories].corr().fillna(0.0)
    for a, b in combinations(categories, 2):
        G.add_edge(a, b, weight=max(corr.loc[a, b], 0))
    pos = networkx.circular_layout(G)
    sizes, alphas = [], []
    for n in G.nodes:
        prev = G.nodes[n]["prevalence"]
        sizes.append(prev * 8000 + 200)
        alphas.append(1.0 if n == focus else 0.4)
    networkx.draw_networkx_nodes(G, pos, node_color="#FFC0CB",
                                 node_size=sizes, ax=ax,
                                 edgecolors="#A06070", linewidths=0.5,
                                 alpha=alphas)
    networkx.draw_networkx_labels(G, pos, font_size=8, ax=ax)
    edge_widths = []
    edge_alphas = []
    for u, v in G.edges:
        w = G[u][v]["weight"]
        if u == focus or v == focus:
            edge_widths.append(w * 8)
            edge_alphas.append(0.9)
        else:
            edge_widths.append(w * 1.5)
            edge_alphas.append(0.15)
    networkx.draw_networkx_edges(G, pos, width=edge_widths,
                                 edge_color="#666666", alpha=edge_alphas, ax=ax)
    ax.set_title(title or f"{focus} egonet")
    ax.axis("off")

## Semantic egonets…

![Carly Simon "You're So Vain" album cover.](img/p139-i0.png)

## Solitude egonet

In [ ]:
cat = next((c for c in ny_cats.values() if c.lower().startswith("solitude")), None)
if cat:
    egonet(cat, title=f"Solitude egonet")
else:
    print("no solitude category in nietzsche.dic")

## Modesty egonet

In [ ]:
cat = next((c for c in ny_cats.values() if c.lower().startswith("modesty")), None)
if cat:
    egonet(cat, title=f"Modesty egonet")
else:
    print("no modesty category in nietzsche.dic")

## Humility egonet

In [ ]:
cat = next((c for c in ny_cats.values() if c.lower().startswith("humility")), None)
if cat:
    egonet(cat, title=f"Humility egonet")
else:
    print("no humility category in nietzsche.dic")

## Terms especially associated with 'virtue' [`tugend*`] in the Nietzsche corpus

![Google Translate showing German terms (eurer, errreichbar, tiktak, anstellige, diene, unmöglichen, fäuste, prokrustes-bett, renaissance-stile, untugend, verflogene, virtu, eure, gerechtigkeit, brüllen, lastern, schenkende, grossmuth, sokratischen) and their English translations.](img/p143-i0.png)

## Before we end, sticky note feedback

- On the green sticky note, write one thing that went well this session.
- On the red sticky note, write one thing that we can improve on.

# Thanks for your attention!